In [1]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainerCallback,
)
from peft import (
    LoraConfig,
    get_peft_model,
)
import json
from nltk.translate.bleu_score import sentence_bleu
import numpy as np
import os, torch, wandb
from dotenv import load_dotenv
import datetime
load_dotenv()
from datasets import load_dataset, Dataset, load_from_disk
from trl import SFTTrainer, SFTConfig, DataCollatorForCompletionOnlyLM
from huggingface_hub import login
from setproctitle import setproctitle
import time, datetime
import weave
import signal
import logging

/vol/csedu-nobackup/project/lzandbergen-storage/venv2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [58]:
setproctitle("lars-fine-tuning")
os.environ["WANDB_SILENT"] = "true"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

base_model = "/vol/csedu-nobackup/project/lzandbergen-storage/lars-zandbergen-pdf-extraction/models/meta-llama/Llama-3.1-8B-Instruct"
# base_model = "meta-llama/Llama-3.2-1B-Instruct"
max_input_length = 768 # Determined by sequence analysis notebook
max_response_length = 100
project_name = "fine-tuning-experimentation"
debug_level = logging.INFO

# Configure logging
def setup_logging(run_name):
    logging.basicConfig(
        level=debug_level,
        format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
        handlers=[
            logging.StreamHandler()
        ]
    )


# Update the run name and subset size
timestamp = datetime.datetime.now().strftime("%d-%m-%H:%M")
run_base_name = f"test"
subset_size = 20
new_model = f"models/fine-tuned/{run_base_name}"

setup_logging(run_base_name)
logger = logging.getLogger(__name__)

hf_token = os.getenv("HUGGINGFACE_TOKEN")
login(token=hf_token)

wb_key = os.getenv("WANDB_KEY")
wandb.login(key=wb_key)

logger.info(f"New model path: {new_model}")

run = wandb.init(
    project=project_name,
    job_type='fine-tuning',
    dir="/vol/csedu-nobackup/project/lzandbergen-storage/wandb",
    anonymous='allow',
    name=run_base_name
)
weave.init(project_name=project_name)

logger.info(f"Current system time and date: {datetime.datetime.now()}")
logger.info("Start loading the tokenizer")
start_time = time.time()
tokenizer = AutoTokenizer.from_pretrained(base_model)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
# tokenizer.chat_template = None
logger.info(f"End loading the tokenizer time: {time.time() - start_time}")

logger.info("Start loading the dataset")
start_time = time.time()

dataset = load_from_disk("data/MAVE/dataset")
train_dataset = dataset['train'].shuffle(seed=42).select(range(subset_size))
test_dataset = dataset['test'].shuffle(seed=42).select(range(int(subset_size*0.2)))

# Sample from dataset
logger.debug(f"Sample from training dataset: {train_dataset[0]}")

logger.info(f"Loaded dataset, time:{time.time() -start_time}")

# TODO, upgrade to prompt storing.
logger.info("Using new lines (no JSON) prompt.")
system_message = """You are an expert in extracting product features from technical documents. \n
The user will provide a template showing the features they'd like to extract.
Your task is to fill in the "values" for each feature from the provided product information.
If a feature cannot be found, fill in "None" for its value. 
Only answer in newline-formatted responses. \n"""

# Define the function to format the chat template
def format_chat_template(dataset):
    formatted_texts = []

    combined_text = "\n".join([para['text'] for para in dataset['paragraphs']])
    
    # Generate feature names template
    feature_names = [f'    "{attr["key"]}" : "value_{i+1}"' for i, attr in enumerate(dataset['labels'])]
    feature_names = "\n".join(feature_names)

    labels = [f'{lbl["key"].lower()} : {lbl["value"].lower()}' for lbl in dataset['labels']]
    labels = "\n".join(labels)

    user_message = f"""Here is the product information:
        \n\n{combined_text}

Provide the output in new lines format with the feature names as keys and their corresponding "values",
use the template below, where each feature is on a new line. Only respond with the features that are requested.:

{feature_names}\n """

    
    text = f'### Question: \n{system_message} \n{user_message} \n\n### Answer: \n{labels}'
    # prompt = tokenizer.apply_chat_template(text, padding="max_length", max_length=max_input_length, tokenize=False, return_tensors=False)
    
    formatted_texts.append(text)
    
    return {"text": formatted_texts}

# initial_token_count = tokenizer.vocab_size
response_template = "\n### Answer: \n"
response_template_ids = tokenizer.encode(response_template, add_special_tokens=False)[2:]
# added_token_count = tokenizer.add_special_tokens({"additional_special_tokens": [response_template]})
collator = DataCollatorForCompletionOnlyLM(response_template_ids, tokenizer=tokenizer)

train_dataset = train_dataset.map(format_chat_template)
test_dataset = test_dataset.map(format_chat_template)
logger.info("Converted to chat template")

2025-01-30 11:03:16,020 - __main__ - INFO - New model path: models/fine-tuned/test
2025-01-30 11:03:38,333 - __main__ - INFO - Current system time and date: 2025-01-30 11:03:38.333483
2025-01-30 11:03:38,336 - __main__ - INFO - Start loading the tokenizer
2025-01-30 11:03:38,618 - __main__ - INFO - End loading the tokenizer time: 0.28157591819763184
2025-01-30 11:03:38,619 - __main__ - INFO - Start loading the dataset
2025-01-30 11:04:37,179 - __main__ - INFO - Loaded dataset, time:58.55882668495178
2025-01-30 11:04:37,181 - __main__ - INFO - Using new lines (no JSON) prompt.
2025-01-30 11:04:37,200 - __main__ - INFO - Converted to chat template


In [60]:
text = (train_dataset[0]["text"])
new_tokenizer = AutoTokenizer.from_pretrained(base_model, truncation=False)
# new_tokenizer.add_special_tokens({"additional_special_tokens": [response_template]})
new_tokenizer.pad_token = new_tokenizer.eos_token
tokenized_text = new_tokenizer(text, padding="max_length", max_length=max_input_length, return_tensors="pt", truncation=True)

string = ""
for token_id, token in zip(tokenized_text['input_ids'][0], new_tokenizer.convert_ids_to_tokens(tokenized_text['input_ids'][0])):
    string += f"{token}({token_id}), "

def print_tokens_with_ids(txt):
    tokens = new_tokenizer.tokenize(txt)
    ids = new_tokenizer.convert_tokens_to_ids(tokens)
    print(list(zip(tokens, ids)))

collated_text = collator(train_dataset[0]["text"])

AttributeError: 'str' object has no attribute 'size'

In [ ]:
# Store a sample in weave as an example of the prompt
sample_prompt = (train_dataset[0]["text"])
weave.publish(sample_prompt, "example prompt")
weave.publish(system_message, "System prompt")

logger.info("Got past the dataset mapping")

def tokenize_function(dataset):
    # TODO: Improve this function to handle multiple texts
    texts = [text if isinstance(text, str) else text[0] if isinstance(text, list) else str(text) 
            for text in dataset['text']]

    # Tokenize with padding and truncation
    result = tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=max_input_length, # Determined by sequence_analysis notebook
        return_tensors=None  # Return list instead of tensors
    )
    
    dataset['text'] = result['input_ids'].copy()
    return result

tokenized_train = train_dataset.map(tokenize_function, batched=True, num_proc=4)
tokenized_train = tokenized_train.remove_columns([col for col in tokenized_train.column_names if col not in ['attention_mask', 'input_ids']])
tokenized_test = test_dataset.map(tokenize_function, batched=True, num_proc=4)
tokenized_test = tokenized_test.remove_columns([col for col in tokenized_test.column_names if col not in ['attention_mask', 'input_ids']])

torch_dtype = torch.float16
attn_implementation = "eager"

📦 Published to https://wandb.ai/lzandbergen-ru/fine-tuning-experimentation/weave/objects/example-prompt/versions/iVGjjv4cVa4hAxqvpr4N3mAjdBvuZ8cdcqn1RCdbHgA


2025-01-30 09:55:27,763 - __main__ - INFO - Got past the dataset mapping


📦 Published to https://wandb.ai/lzandbergen-ru/fine-tuning-experimentation/weave/objects/System-prompt/versions/Omi8ik9dc2ORTrHPQQgGBy4X9sB5lqP2FbX6fuiqMXg


Map (num_proc=4): 100%|██████████| 4/4 [00:00<00:00, 21.65 examples/s]


In [11]:
# Add this before training:
test_sample = tokenized_train[0]["input_ids"]
decoded_sample = tokenizer.decode(test_sample)
logger.info(f"Decoded sample:\n{decoded_sample}")
print(tokenizer.decode(response_template_ids))

# Verify the response template appears in decoded text
# assert response_template_ids in test_sample, "Response template not found in sample"

2025-01-30 10:00:16,043 - __main__ - INFO - Decoded sample:
<|begin_of_text|>### Question: 
You are an expert in extracting product features from technical documents. 

The user will provide a template showing the features they'd like to extract.
Your task is to fill in the "values" for each feature from the provided product information.
If a feature cannot be found, fill in "None" for its value. 
Only answer in newline-formatted responses. 
 
Here is the product information:
        

Biviel Women's Bv2720 Boot
Biviel is the creation of a great team of shoemakers and designers. It is the young and more affordable collection from the owner and designer of Blay. The shoes are designed as "feminine relaxed" to be flexible and comfortable but always elegant. The leathers are the softest Italian calfskins- the soles are rubber imported from Germany. They have the feel as well as the look of Italian shoes twice the price.
100% Leather
Rubber sole
Shaft measures approximately 10" from arch
H

 Answer: 



In [74]:
sample = tokenized_train[0]

collator_response = collator.torch_call([sample])
# print(collator_response)
# print(collator_response['input_ids'])
print(list(zip(collator_response['input_ids'][0], tokenizer.convert_ids_to_tokens(collator_response['input_ids'][0]), collator_response['attention_mask'][0], collator_response['labels'][0])))
def print_collator_response(collator_response):
    for input_id, token, attention_mask, label in zip(
        collator_response['input_ids'][0], 
        tokenizer.convert_ids_to_tokens(collator_response['input_ids'][0]), 
        collator_response['attention_mask'][0], 
        collator_response['labels'][0]
    ):
        print(f"Text: '{token}', Token ID: {input_id}, Attention Mask: {attention_mask}, Label: {label}")

print_collator_response(collator_response)

print(tokenizer.decode(sample['input_ids']))
print(response_template)
print(tokenizer.tokenize(response_template))
print(response_template_ids)

[(tensor(128000), '<|begin_of_text|>', tensor(1), tensor(-100)), (tensor(14711), '###', tensor(1), tensor(-100)), (tensor(16225), 'ĠQuestion', tensor(1), tensor(-100)), (tensor(25), ':', tensor(1), tensor(-100)), (tensor(720), 'ĠĊ', tensor(1), tensor(-100)), (tensor(2675), 'You', tensor(1), tensor(-100)), (tensor(527), 'Ġare', tensor(1), tensor(-100)), (tensor(459), 'Ġan', tensor(1), tensor(-100)), (tensor(6335), 'Ġexpert', tensor(1), tensor(-100)), (tensor(304), 'Ġin', tensor(1), tensor(-100)), (tensor(60508), 'Ġextracting', tensor(1), tensor(-100)), (tensor(2027), 'Ġproduct', tensor(1), tensor(-100)), (tensor(4519), 'Ġfeatures', tensor(1), tensor(-100)), (tensor(505), 'Ġfrom', tensor(1), tensor(-100)), (tensor(11156), 'Ġtechnical', tensor(1), tensor(-100)), (tensor(9477), 'Ġdocuments', tensor(1), tensor(-100)), (tensor(13), '.', tensor(1), tensor(-100)), (tensor(4815), 'ĠĊĊ', tensor(1), tensor(-100)), (tensor(791), 'The', tensor(1), tensor(-100)), (tensor(1217), 'Ġuser', tensor(1), t

In [77]:
logger.info(f"Response template: {response_template}")
logger.info(f"Response template IDs: {response_template_ids}")
sample_input_ids = tokenized_train[0]['input_ids']
end_index = sample_input_ids.index(tokenizer.eos_token_id)
logger.info(f"Sample input IDs (first 50 tokens): {sample_input_ids[-50:]}")

2025-01-30 11:20:56,343 - __main__ - INFO - Response template: 
### Answer: 

2025-01-30 11:20:56,345 - __main__ - INFO - Response template IDs: [22559, 25, 720]


AttributeError: 'list' object has no attribute 'split'

In [ ]:

# Clear cache to make sure enough memory is available
torch.cuda.empty_cache()

# Load model
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    local_files_only=True, 
    device_map = "cuda:0"
    # attn_implementation=attn_implementation
)
logger.info("Model loaded")

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    task_type="causal_lm",
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj']
)
model = get_peft_model(model, peft_config)

def amazon_metrics(preds: list[dict], labels: list[dict]):
    # Initialize trackers
    exact_matches = 0
    total_attributes = 0
    bleu_scores = []
    jaccard_scores = []
    
    # Iterate over each sample
    for pred_dict, label_dict in zip(preds, labels):

        logger.debug(f"Running evaluation on sample:Pred: {pred_dict}, Label: {label_dict}")
        # Check all attributes for this sample
        for attr in label_dict.keys():
            total_attributes += 1
            pred_value = pred_dict.get(attr, "").strip().lower()
            true_value = label_dict.get(attr, "").strip().lower()
            
            # Exact match (case-insensitive)
            if pred_value == true_value:
                exact_matches += 1
            
            # BLEU score (skip if either is empty)
            if pred_value and true_value:
                pred_tokens = pred_value.split()
                true_tokens = true_value.split()
                try:
                    bleu = sentence_bleu([true_tokens], pred_tokens, weights=(0.5, 0.5))
                    bleu_scores.append(bleu)
                except:
                    bleu_scores.append(0.0)
                
                # Jaccard similarity
                pred_set = set(pred_tokens)
                true_set = set(true_tokens)
                intersection = len(pred_set & true_set)
                union = len(pred_set | true_set)
                jaccard = intersection / union if union > 0 else 0.0
                jaccard_scores.append(jaccard)
    
    # Aggregate metrics
    return {
        "exact_match_ratio": exact_matches / total_attributes if total_attributes > 0 else 0.0,
        "avg_bleu": np.mean(bleu_scores) if bleu_scores else 0.0,
        "avg_jaccard": np.mean(jaccard_scores) if jaccard_scores else 0.0,
        "total_samples": len(preds),
        "total_attributes": total_attributes
    }

def compute_metrics(eval_preds):    
    preds, labels = eval_preds
    return amazon_metrics(preds, labels)

@weave.op()
class WeaveCallback(TrainerCallback):
    def __init__(self):
        self.step = 0
        
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            # Training metrics
            if "train_loss" in logs:
                wandb.log({"train/loss": logs["train_loss"],
                            "train/learning_rate": logs["learning_rate"],
                            "train/epoch": logs["epoch"],
                            "train/grad_norm": logs["grad_norm"]}
                            )
                            
            # Evaluation metrics
            if "eval_exact_match_ratio" in logs:
                wandb.log({
                    "eval/exact_match_ratio": logs["eval_exact_match_ratio"],
                    "eval/avg_bleu": logs["eval_avg_bleu"],
                    "eval/avg_jaccard": logs["eval_avg_jaccard"]
                })


training_arguments = SFTConfig(
    max_seq_length=max_input_length + max_response_length,
    output_dir=new_model,
    auto_find_batch_size=True,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    optim="paged_adamw_32bit",
    num_train_epochs=1,
    # num_train_epochs=4, # Using 1 epoch might prevent overfitting as per: https://magazine.sebastianraschka.com/p/practical-tips-for-finetuning-llms
    eval_strategy="steps",
    # eval_steps=500,
    eval_steps=10,
    save_strategy="steps",
    save_steps=1000,
    save_total_limit=3,
    logging_steps=50,
    logging_strategy="steps",
    learning_rate=2e-4,
    fp16=True,
    bf16=False,
    report_to="wandb",
    dataloader_num_workers=4,
    run_name=run_base_name,
)

torch.cuda.empty_cache()
logger.info(f"Run Name: {training_arguments.run_name}")

model = model.to("cuda:0")
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    peft_config=peft_config,
    args=training_arguments,
    packing=False,
    callbacks=[WeaveCallback()],
    data_collator=collator,
    compute_metrics=compute_metrics
)

def save_model_on_signal():
    logger.info(f"Signal received, saving model to {new_model}")
    trainer.save_model(new_model)
    model.save_pretrained(new_model)

# Register the signal handler
signal.signal(12, save_model_on_signal)

try:
    trainer.train()
    trainer.save_model(new_model)
    model.save_pretrained(new_model)
except Exception as e:
    logger.info(f"Exception occurred: {e}")
    save_model_on_signal()

weave.finish()